# 02 探索性数据分析（EDA）与数据清洗
**论文章节对应**：第3章 数据处理

本notebook完成以下任务：
1. 描述性统计与缺失值分析
2. K-S检验验证log变换必要性（图4-1）
3. 数据清洗：去重 + IQR异常值处理 + 缺失值报告
4. 区域造价差异可视化（图4-2）
5. 数据来源与年份分布统计（图4-3）

**输入**：`data/raw/housing_cost_raw.csv`
**输出**：`data/processed/housing_cost_clean.csv` + 图4-1/4-2/4-3

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.feature_engineering import (
    ks_normality_test, compare_log_transform,
    remove_outliers_iqr, dedup_by_key_fields, report_missing
)

# 全局图表设置
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
DPI = 300

# 读取原始数据
df = pd.read_csv('data/raw/housing_cost_raw.csv', encoding='utf-8-sig')
print(f'原始数据: {df.shape[0]} 条, {df.shape[1]} 个字段')
df.head()

## 2.1 描述性统计

In [ ]:
print('=== 描述性统计 ===')
print(df.describe().round(2))

In [ ]:
print('=== 数据类型 ===')
print(df.dtypes)
print('\n=== 各字段缺失情况 ===')
report_missing(df)

## 2.2 K-S检验：验证log变换必要性（图4-1）

In [ ]:
print('=== K-S正态性检验 ===')

# 原始造价K-S检验
stat_orig, p_orig = ks_normality_test(df['unit_cost'].dropna(), '原始单方造价')

# log1p变换后K-S检验
stat_log, p_log = ks_normality_test(np.log1p(df['unit_cost'].dropna()), 'log1p变换后造价')

print(f'\n结论：')
if p_orig <= 0.05 and p_log > p_orig:
    print(f'  原始造价不满足正态分布（p={p_orig:.4f} ≤ 0.05），')
    print(f'  log1p变换后p值提升（p={p_log:.4f}），验证了对目标变量进行log变换的必要性')
else:
    print(f'  原始造价 p={p_orig:.4f}，log变换后 p={p_log:.4f}')

In [ ]:
# 绘制图4-1：分布对比图
compare_log_transform(
    df['unit_cost'].dropna(),
    col_name='单方造价',
    save_path='outputs/figures/fig4_1_distribution.png'
)

## 2.3 数据清洗

In [ ]:
print(f'清洗前: {len(df)} 条')

# Step 1: 去除目标变量缺失
df = df.dropna(subset=['unit_cost'])
print(f'去除unit_cost缺失后: {len(df)} 条')

# Step 2: 去除明显异常值（造价<500 或 >15000 元/㎡为异常）
before = len(df)
df = df[(df['unit_cost'] >= 500) & (df['unit_cost'] <= 15000)]
print(f'去除造价极端异常值后: {len(df)} 条（移除{before - len(df)}条）')

# Step 3: IQR方法去除造价异常值
df = remove_outliers_iqr(df, 'unit_cost', factor=2.5)  # 使用2.5倍IQR，较宽松

# Step 4: 四字段联合去重
df = dedup_by_key_fields(df,
    key_cols=['province', 'completion_year', 'total_area', 'above_floors'])

# Step 5: 去除层数逻辑错误
before = len(df)
df = df[df['above_floors'] >= 1]  # 至少1层
print(f'去除层数异常后: {len(df)} 条（移除{before - len(df)}条）')

# Step 6: 确保completion_year在合理范围
before = len(df)
df = df[df['completion_year'].between(2020, 2025)]
print(f'过滤年份范围后: {len(df)} 条（移除{before - len(df)}条）')

print(f'\n清洗完成，最终: {len(df)} 条')

In [ ]:
# 重置索引
df = df.reset_index(drop=True)

# 缺失值报告（清洗后）
print('=== 清洗后缺失值报告 ===')
report_missing(df)

## 2.4 各省份造价箱线图（图4-2）

In [ ]:
# 计算各省中位造价（排序）
province_median = df.groupby('province')['unit_cost'].median().sort_values(ascending=False)
top_provinces = province_median.index.tolist()

# 限制展示省份数量（最多18个）
show_provinces = top_provinces[:18]
df_plot = df[df['province'].isin(show_provinces)]

fig, ax = plt.subplots(figsize=(14, 7))

# 按中位造价排序绘制箱线图
data_by_province = [df_plot[df_plot['province'] == p]['unit_cost'].values
                    for p in show_provinces]

bp = ax.boxplot(data_by_province, labels=show_provinces,
                patch_artist=True, notch=False,
                medianprops={'color': 'red', 'linewidth': 2})

# 渐变颜色
colors = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, len(show_provinces)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels(show_provinces, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('单方造价（元/㎡）', fontsize=12)
ax.set_title('图4-2 各省份单方造价箱线图（按中位数降序）', fontsize=13)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/fig4_2_province_boxplot.png', dpi=DPI, bbox_inches='tight')
plt.show()
print('图4-2 已保存至 outputs/figures/fig4_2_province_boxplot.png')

## 2.5 数据来源与年份分布（图4-3）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左图：年份分布（按数据来源堆叠）
if 'data_source' in df.columns:
    year_source = df.groupby(['completion_year', 'data_source']).size().unstack(fill_value=0)
    year_source.plot(kind='bar', stacked=True, ax=axes[0],
                    colormap='Set2', edgecolor='white', width=0.7)
    axes[0].legend(title='数据来源', fontsize=9)
else:
    year_counts = df['completion_year'].value_counts().sort_index()
    axes[0].bar(year_counts.index.astype(str), year_counts.values,
               color='#4e79a7', alpha=0.8, edgecolor='white')

axes[0].set_title('各年份数据量分布', fontsize=12)
axes[0].set_xlabel('竣工年份', fontsize=11)
axes[0].set_ylabel('记录数', fontsize=11)
axes[0].tick_params(axis='x', rotation=0)

# 右图：结构类型分布
if 'structure_type' in df.columns:
    struct_counts = df['structure_type'].value_counts()
    colors_pie = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']
    wedges, texts, autotexts = axes[1].pie(
        struct_counts.values,
        labels=struct_counts.index,
        autopct='%1.1f%%',
        colors=colors_pie[:len(struct_counts)],
        startangle=90,
    )
    for autotext in autotexts:
        autotext.set_fontsize(9)
    axes[1].set_title('结构类型分布', fontsize=12)

plt.suptitle('图4-3 数据集概况统计', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/fig4_3_data_overview.png', dpi=DPI, bbox_inches='tight')
plt.show()
print('图4-3 已保存至 outputs/figures/fig4_3_data_overview.png')

## 2.6 保存清洗后数据

In [ ]:
df.to_csv('data/processed/housing_cost_clean.csv', index=False, encoding='utf-8-sig')

print('=== 清洗后数据摘要 ===')
print(f'✓ 已保存至 data/processed/housing_cost_clean.csv')
print(f'  记录数: {len(df)}')
print(f'  字段数: {len(df.columns)}')
print(f'  年份范围: {df["completion_year"].min()} ~ {df["completion_year"].max()}')
print(f'  省份数: {df["province"].nunique()}')
print(f'  单方造价: {df["unit_cost"].min():.0f} ~ {df["unit_cost"].max():.0f} 元/㎡')
print(f'  均值: {df["unit_cost"].mean():.0f} 元/㎡')
print(f'  中位数: {df["unit_cost"].median():.0f} 元/㎡')